# Capitolo 8 — Il cuscinetto che non si è ancora guastato (§ 8.4)
Autoencoder sullo spettro, addestrato **solo** sul cuscinetto sano; i guasti compaiono solo nel test. Poi il limite (sano a 1 HP) e il rimedio.

In [1]:
import sys; sys.path.insert(0, "..")
from utils import fissa_seme
import dati
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
fissa_seme(42)

from sklearn.metrics import roc_auc_score
L, PASSO = 1024, 512; CLASSI, NOMI = dati.CWRU_CLASSI, dati.CWRU_NOMI
def finestre_classe(seg, c, a, b):
    s = seg[c]; n = len(s); s = s[int(n * a):int(n * b)]; return np.stack([s[i:i + L] for i in range(0, len(s) - L + 1, PASSO)])
def spettro(X):
    X = dati.normalizza_finestre(X); return np.log1p(np.abs(np.fft.rfft(X, axis=1))[:, :L // 2]).astype(np.float32)

class AutoencoderSpettro(nn.Module):
    def __init__(self, d=8):
        super().__init__(); self.enc = nn.Sequential(nn.Linear(512, 64), nn.ReLU(), nn.Linear(64, d)); self.dec = nn.Sequential(nn.Linear(d, 64), nn.ReLU(), nn.Linear(64, 512))
    def forward(self, x): z = self.enc(x); return self.dec(z), z

def addestra_rilevatore(S_tr, S_va, epoche=200):
    media, dev = S_tr.mean(0), S_tr.std(0) + 1e-8; nz = lambda S: torch.from_numpy((S - media) / dev)
    fissa_seme(42); ae = AutoencoderSpettro(); opt = torch.optim.Adam(ae.parameters(), lr=1e-3, weight_decay=1e-5); fn = nn.MSELoss(); Xs, Xv = nz(S_tr), nz(S_va); migliore = (float("inf"), None)
    for epoca in range(epoche):
        ae.train(); perm = torch.randperm(len(Xs))
        for i in range(0, len(Xs), 32):
            xb = Xs[perm[i:i + 32]]; opt.zero_grad(); fn(ae(xb)[0], xb).backward(); opt.step()
        ae.eval()
        with torch.no_grad(): lv = fn(ae(Xv)[0], Xv).item()
        if lv < migliore[0]: migliore = (lv, {k: v.clone() for k, v in ae.state_dict().items()})
    ae.load_state_dict(migliore[1]); ae.eval()
    def errori(S):
        with torch.no_grad(): X = nz(S); return ((ae(X)[0] - X) ** 2).mean(1).numpy()
    return errori

## Addestrato sul sano a 0 HP

In [2]:
seg0 = dati.cuscinetti(0)
S_tr = spettro(finestre_classe(seg0, "NORMAL", 0, 0.7)); S_va = spettro(finestre_classe(seg0, "NORMAL", 0.7, 0.85)); S_te = spettro(finestre_classe(seg0, "NORMAL", 0.85, 1.0))
guasti = {c: spettro(finestre_classe(seg0, c, 0.85, 1.0)) for c in CLASSI if c != "NORMAL"}
errori = addestra_rilevatore(S_tr, S_va)
soglia = np.percentile(errori(S_va), 99); e_ok = errori(S_te); e_g = {c: errori(S) for c, S in guasti.items()}
print(f"soglia {soglia:.2f} | sano test: mediana {np.median(e_ok):.2f}, oltre soglia {(e_ok > soglia).mean():.0%}")
for c, e in e_g.items(): print(f"  {NOMI[c]:16s} mediana {np.median(e):6.1f}   rilevato {(e > soglia).mean():.0%}")
tutti = np.concatenate(list(e_g.values())); print("AUC sano/guasto:", round(roc_auc_score(np.r_[np.zeros(len(e_ok)), np.ones(len(tutti))], np.r_[e_ok, tutti]), 3))

scarico cwru_0hp.csv ... 

35.5 MB


soglia 1.20 | sano test: mediana 0.93, oltre soglia 0%
  sfera 0.18 mm    mediana   28.5   rilevato 100%
  sfera 0.36 mm    mediana   19.3   rilevato 100%
  sfera 0.53 mm    mediana   29.6   rilevato 100%
  pista int. 0.18  mediana   11.9   rilevato 100%
  pista int. 0.36  mediana   31.9   rilevato 100%
  pista int. 0.53  mediana   53.4   rilevato 100%
  pista est. 0.18  mediana   52.7   rilevato 100%
  pista est. 0.36  mediana   23.8   rilevato 100%
  pista est. 0.53  mediana   42.6   rilevato 100%
AUC sano/guasto: 1.0


## Il limite: il sano a 1 HP visto dal rilevatore a 0 HP

In [3]:
seg1 = dati.cuscinetti(1)
S1_ok = spettro(finestre_classe(seg1, "NORMAL", 0, 1.0)); e1_ok = errori(S1_ok)
print(f"sano a 1 HP: mediana {np.median(e1_ok):.2f}, segnalato come anomalo {(e1_ok > soglia).mean():.0%}")

scarico cwru_1hp.csv ... 

75.2 MB


sano a 1 HP: mediana 3.05, segnalato come anomalo 100%


## Il rimedio: il normale deve contenere tutte le condizioni

In [4]:
S_tr2 = np.concatenate([S_tr, spettro(finestre_classe(seg1, "NORMAL", 0, 0.7))]); S_va2 = np.concatenate([S_va, spettro(finestre_classe(seg1, "NORMAL", 0.7, 0.85))])
errori2 = addestra_rilevatore(S_tr2, S_va2); soglia2 = np.percentile(errori2(S_va2), 99)
e_ok1 = errori2(spettro(finestre_classe(seg1, "NORMAL", 0.85, 1.0))); e_g1 = np.concatenate([errori2(spettro(finestre_classe(seg1, c, 0.85, 1.0))) for c in CLASSI if c != "NORMAL"])
print(f"addestrato su 0+1 HP, soglia {soglia2:.2f}: sano 1 HP falsi allarmi {(e_ok1 > soglia2).mean():.0%} | guasti 1 HP rilevati {(e_g1 > soglia2).mean():.0%} | AUC {roc_auc_score(np.r_[np.zeros(len(e_ok1)), np.ones(len(e_g1))], np.r_[e_ok1, e_g1]):.2f}")

addestrato su 0+1 HP, soglia 1.06: sano 1 HP falsi allarmi 0% | guasti 1 HP rilevati 100% | AUC 1.00
